# 03 · Generalisation to new topologies (ToBaCCo)
Reproduces report sections 6.1–6.3: 12 generated MOFs on nets absent from hMOF, simulated with both conventions and compared with the graph-network predictions (Spearman 0.98; ultra-light frameworks about +4 g/L).

The 894-structure screen itself is optional below (generating structures with ToBaCCo takes several hours). **Runtime: about 2 hours for the checks.** **Attach:** the `mofh2_pipeline` dataset.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['03_topologies']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

### Featurise the packaged generated structures

In [ ]:
!python featurise_t130.py

### Run simulations: `03_topologies` (36 runs)

In [ ]:
!bash rq_any.sh queues/03_topologies.txt {MAX_HOURS}

In [ ]:
!python compare_topo.py && python compare_topo_m2.py

### Optional: regenerate and screen all ToBaCCo structures
Set `RUN_TOBACCO = True`. Generates MOFs for each template in `tob_test/selected.txt` (120 s limit each), then featurises and predicts every structure with `predict_topo.py`.

In [ ]:
RUN_TOBACCO = False
if RUN_TOBACCO:
    !cd tob_test && pip install -q -r requirements.txt && cat selected.txt | sed 's/\.cif$//' | xargs -P $NPROC -I{} bash run_one.sh {}
    !python predict_topo.py

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG topo_gcmc_validation.csv topo_gcmc_validation_m2.csv

In [ ]:
!python stage_io.py save stage03_outputs
# Attach /kaggle/working/stage03_outputs.tgz (via 'Save Version' output) as input to later notebooks.